# Poker says skill shows up in 1,500 hands. Here is why you need 129,600

**Out of Sample** — [read the article](https://arthurnoo.pythonanywhere.com/poker-and-strategies)

This notebook reproduces every figure and every number in the article.

## How to run it

| | |
| --- | --- |
| **Run it, nothing to install** | [Open in Colab](https://colab.research.google.com/github/Arthurnoo/out-of-sample/blob/main/notebooks/poker-and-strategies.ipynb) |
| **Open it in VS Code** | [Open in VS Code](https://vscode.dev/github/Arthurnoo/out-of-sample/blob/main/notebooks/poker-and-strategies.ipynb) |

Colab runs it in your browser. VS Code opens the real editor, also in your
browser, with nothing to install. Either way the next cell installs whatever is
missing, so you can simply **Run All**. The first run downloads prices and takes
a minute or two.

On your own machine instead: `git clone https://github.com/Arthurnoo/out-of-sample`

## How to read it

Each section matches a section of the article. Run a cell, look at what it
prints, then read the note underneath explaining what to make of it.

**Section 0 holds every parameter you might want to change.** Everything after
it is written so that changing those values changes the whole notebook. That is
the point: the numbers below are mine, and the interesting question is what
happens to them when you move something.

Your numbers may differ slightly from the article if you run this later. The
data keeps growing, which is itself worth noticing.

## If you find a mistake

Tell me. There is a form at the bottom of the article, and a wrong number
corrected in public is worth more than a right one nobody checked.

In [ ]:
# Installs anything missing, then does nothing on later runs.
# This is what makes "Run All" work in Colab, in VS Code, or on a bare machine.
import importlib, subprocess, sys

for package in ['yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy']:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f'installing {package} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

print('ready')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

pd.set_option('display.width', 130)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '#e4e9ee', 'axes.facecolor': 'white'})

## 0. The knobs

Two numbers drive every table below. Put your own in, or those of a strategy
you have measured, and everything recomputes.

| Knob | What it does | Try |
| --- | --- | --- |
| `WINRATE` | big blinds won per 100 hands | `2`, `10` |
| `SD` | standard deviation on the same unit | `70`, `110` |
| `HANDS_PER_YEAR` | how much you play | `50_000`, `200_000` |
| `BB_PER_BUYIN` | big blinds in one buy-in | `100` |
| `PLAYERS` | players in the simulation | lower it to go faster |

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  PARAMETERS — this is the only cell you need to edit
# ─────────────────────────────────────────────────────────────────────

WINRATE        = 5.0        # big blinds per 100 hands
SD             = 90.0       # standard deviation, same unit
HANDS_PER_YEAR = 100_000
BB_PER_BUYIN   = 100
PLAYERS        = 5000

BLOCK = 100                 # a 'block' is 100 hands, the unit used throughout

## 1. A player is a strategy

The ratio of the two numbers is a Sharpe ratio, measured per block of 100
hands. Annualising it means multiplying by the square root of the number of
blocks played in a year, exactly as a daily Sharpe is annualised by √252.

In [ ]:
ratio = WINRATE / SD                     # Sharpe par bloc de 100 mains
print(f"edge over noise, per block: {ratio:.3f}")

for h in (50_000, 100_000, 200_000):
    blocks = h / BLOCK
    print(f"{h:>8,} hands a year -> annual Sharpe {ratio*np.sqrt(blocks):.1f}")

## 2. How long before you know

After N blocks the cumulative win has mean `N*WINRATE` and standard deviation
`SD*sqrt(N)`. So the t-statistic is `WINRATE*sqrt(N)/SD`, and inverting it for
a target t gives the sample size:

```
N = (t * SD / WINRATE)**2
```

`p_losing` is the share of players with exactly this skill who still show a
loss after that many hands.

In [ ]:
def spread(hands, winrate=WINRATE, sd=SD):
    """Fourchette du gain MESURE apres `hands` mains, pour un joueur donne."""
    n = hands / BLOCK                     # nombre de blocs
    se = sd / np.sqrt(n)                  # erreur-type du gain mesure
    return {'hands': hands, 'se': se,
            'p05': winrate - 1.645*se, 'p95': winrate + 1.645*se,
            't': winrate/se,
            'p_losing_pct': norm.cdf(-winrate/se)*100}


def hands_for_t(t=2.0, winrate=WINRATE, sd=SD):
    """Mains necessaires pour atteindre un t donne."""
    return (t*sd/winrate)**2 * BLOCK


tab = pd.DataFrame([spread(h) for h in
                    (5_000, 10_000, 25_000, 50_000, 100_000, 200_000, 500_000)])
print(tab.to_string(index=False, float_format=lambda v: f'{v:12.2f}'))
print(f"\nhands for t = 2: {hands_for_t():,.0f}")
print(f"that is {hands_for_t()/HANDS_PER_YEAR:.2f} years at {HANDS_PER_YEAR:,} hands a year")

In [ ]:
grid = np.logspace(3.3, 6, 160)
rows = pd.DataFrame([spread(h) for h in grid])
fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(rows.hands, rows.p05, rows.p95, color='#2d6fa8', alpha=0.16)
ax.axhline(WINRATE, color='#2d6fa8', lw=1.8)
ax.axhline(0, color='#c83c50', lw=1.3, ls='--')
# Seuil exact : p05 > 0 des que hands depasse (1.645*SD/WINRATE)**2 * BLOCK.
clears_at = (1.645*SD/WINRATE)**2 * BLOCK
ax.axvline(clears_at, color='#0e8f5f', ls=':')
print(f"the 90% band clears zero at {clears_at:,.0f} hands")
print(f"t = 2 needs {hands_for_t():,.0f}, because 2 is a higher bar than 1.645")
ax.set_xscale('log'); ax.set_xlabel('hands played (log scale)')
ax.set_ylabel('measured win rate (bb/100)'); plt.show()

## 3. Why the published threshold is a different number

Potter van Loon, van den Assem and van Dolder (PLOS ONE, 2015) put the point
where skill dominates chance at about **1,500 hands**. Section 2 just gave
129,600. Both are right, because they answer different questions.

A player's observed result is his true level plus noise of size `SD/sqrt(N)`.
The paper asks when a **ranking of many players** reflects their levels, which
happens once the spread of true levels exceeds the spread of the noise:

```
sd_skill**2 > SD**2 / N        so        N > (SD / sd_skill)**2
```

No individual win rate appears in that expression. The answer depends only on
how different the players are from one another.

In [ ]:
def hands_to_rank(sd_skill, sd_luck=SD):
    """Mains avant que le CLASSEMENT d'une population reflete le niveau."""
    return np.inf if sd_skill <= 0 else (sd_luck/sd_skill)**2 * BLOCK


def implied_spread(hands, sd_luck=SD):
    """L'inverse : quel ecart de niveau rend `hands` suffisantes ?"""
    return sd_luck / np.sqrt(hands / BLOCK)


PAPER_HANDS = 1500
spread = implied_spread(PAPER_HANDS)
print(f"the paper's {PAPER_HANDS:,} hands implies a skill spread of {spread:.1f} bb/100")
print(f"check: ({SD:.0f}/{spread:.1f})**2 * {BLOCK} = {hands_to_rank(spread):,.0f} hands\n")

for st in (5, 10, 15, 20, spread, 30):
    print(f"  spread {st:>5.1f} bb/100 -> {hands_to_rank(st):>9,.0f} hands to rank players")

solo = hands_for_t()
print(f"\none player proving his own rate: {solo:,.0f} hands")
print(f"ratio between the two: {solo/PAPER_HANDS:.0f}x")

grid = np.linspace(3, 35, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid, [hands_to_rank(g) for g in grid], lw=2, color='#2d6fa8')
ax.axhline(solo, color='#c83c50', ls='--')
ax.plot(spread, PAPER_HANDS, 'o', color='#0e8f5f', ms=7)
ax.set_yscale('log'); ax.set_xlabel('spread of true skill across players (bb/100)')
ax.set_ylabel('hands needed (log scale)'); plt.show()

## 4. The same equation on the market side

For a strategy, `t = Sharpe * sqrt(years)`, so `years = (t/Sharpe)**2`. It is
the same equation as section 2 written in another unit, and the table below
shows the two side by side for the same person.

In [ ]:
def years_for_t(sharpe, t=2.0):
    """Annees necessaires pour etablir un Sharpe annuel donne."""
    return np.inf if sharpe <= 0 else (t/sharpe)**2


blocks_per_year = HANDS_PER_YEAR / BLOCK
rows = []
for wr in (2.7, 5.0, 7.2, 9.9):
    sharpe = (wr/SD) * np.sqrt(blocks_per_year)
    rows.append({'winrate_bb': wr, 'ratio': wr/SD, 'hands_for_t2': hands_for_t(winrate=wr),
                 'annual_sharpe': sharpe, 'years_for_t2': years_for_t(sharpe)})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:14.3f}'))
print("\nThe last two columns describe the same player as the first two.")

## 5. What one year looks like

Many players of **identical** skill, one year each. Each block of 100 hands is
drawn from a normal distribution: a single hand is nothing like normal, but
over 100 hands the central limit theorem has done its work, which is why the
block is the unit used throughout.

In [ ]:
rng = np.random.default_rng(1)
blocks = int(HANDS_PER_YEAR / BLOCK)
paths = np.cumsum(rng.normal(WINRATE, SD, size=(PLAYERS, blocks)), axis=1)

final = paths[:, -1]
worst_dd = np.min(paths - np.maximum.accumulate(paths, axis=1), axis=1)
print(f"{PLAYERS} identical players, {HANDS_PER_YEAR:,} hands each")
print(f"  median result        {np.median(final)/BB_PER_BUYIN:>7.1f} buy-ins")
print(f"  still losing         {np.mean(final < 0)*100:>7.1f}%")
print(f"  median worst swing   {np.median(worst_dd)/BB_PER_BUYIN:>7.1f} buy-ins")
print(f"  worst swing, p05     {np.percentile(worst_dd, 5)/BB_PER_BUYIN:>7.1f} buy-ins")

x = np.arange(1, blocks+1) * BLOCK
fig, ax = plt.subplots(figsize=(8, 4))
for row in paths[:60]:
    ax.plot(x, row, lw=0.7, color='#8b98a6', alpha=0.35)
ax.fill_between(x, np.percentile(paths, 5, axis=0), np.percentile(paths, 95, axis=0),
                color='#2d6fa8', alpha=0.14)
ax.plot(x, np.median(paths, axis=0), lw=2.2, color='#2d6fa8')
ax.axhline(0, color='#c83c50', lw=1.2)
ax.set_xlabel('hands played'); ax.set_ylabel('cumulative win, big blinds'); plt.show()

## 6. The bankroll rule is Kelly

The position sizing piece gives the optimal fraction as `f* = mu / sd**2`. A
player does not pick a leverage, he picks the stake he sits down at, which is
the same decision. The bankroll that makes that fraction feasible is therefore
`sd**2 / mu`.

The risk of ruin from a bankroll `B` is `exp(-2*mu*B/sd**2)`, the standard
Brownian approximation. It is a probability over an infinite horizon, so it does
not depend on how long you play.

In [ ]:
def kelly_bankroll(winrate=WINRATE, sd=SD):
    """Capital qui rend la fraction de Kelly jouable, en grosses blindes."""
    return sd**2 / winrate


def risk_of_ruin(bankroll_bb, winrate=WINRATE, sd=SD):
    """Probabilite d'atteindre zero un jour, en continuant a jouer."""
    return 1.0 if winrate <= 0 else np.exp(-2*winrate*bankroll_bb / sd**2)


kb = kelly_bankroll()
print(f"Kelly bankroll  {kb:,.0f} bb = {kb/BB_PER_BUYIN:.1f} buy-ins")
print(f"risk of ruin there  {risk_of_ruin(kb)*100:.1f}%\n")
for b in (10, 16.2, 20, 25, 30, 40):
    print(f"{b:>5.1f} buy-ins -> {risk_of_ruin(b*BB_PER_BUYIN)*100:5.1f}% chance of going broke")

buyins = np.linspace(1, 60, 240)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.axvspan(20, 30, color='#0e8f5f', alpha=0.12)
ax.plot(buyins, [risk_of_ruin(b*BB_PER_BUYIN)*100 for b in buyins], lw=2, color='#2d6fa8')
ax.set_xlabel('bankroll, in buy-ins'); ax.set_ylabel('chance of going broke (%)')
ax.set_title('the shaded band is the usual advice', loc='left'); plt.show()

In [ ]:
# Les deux exigences varient comme 1/mu : un joueur deux fois moins bon a
# besoin de deux fois plus de caves et de QUATRE fois plus de mains.
print(pd.DataFrame([
    {'winrate_bb': wr, 'hands_for_t2': hands_for_t(winrate=wr),
     'kelly_buyins': kelly_bankroll(wr)/BB_PER_BUYIN,
     'buyins_5pct_ruin': -np.log(0.05)*SD**2/(2*wr)/BB_PER_BUYIN}
    for wr in (10.0, 5.0, 2.0, 1.0)
]).to_string(index=False, float_format=lambda v: f'{v:17.1f}'))

## 7. Five things worth changing

1. **Set `WINRATE = 2`.** A marginal winner needs 810,000 hands to prove it and
   40 buy-ins to survive. Check how those two change together.
2. **Raise `SD` to 110**, which is closer to a loose game. The edge is
   unchanged and everything else gets worse, because every requirement here
   scales with the square of the standard deviation.
3. **Put a strategy in instead of a player.** Use its return and volatility per
   trade as `WINRATE` and `SD`, and its number of trades a year as
   `HANDS_PER_YEAR`. Every table still applies.
4. **Drop `WINRATE` to a small negative number.** The risk of ruin becomes 1 at
   every bankroll, which is the only thing that genuinely cannot be sized
   around.
5. **Make the player move down a stake** when his bankroll halves, and
   re-measure the risk of ruin. It should fall well below the formula, which
   assumes a fixed stake forever.

Found something I got wrong? There is a form at the bottom of the article.